# Logistic Regression and Expected Goals (xG)

**Chapter 4: Predicting Soccer Outcomes with Classification**

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand how logistic regression works
- Build an Expected Goals (xG) model using logistic regression
- Engineer features from soccer event data
- Interpret model coefficients
- Make predictions with your trained model

## Prerequisites

- Completed Notebook 01 (Introduction to Classification)
- Understanding of probability (0 to 1)
- Basic Python and pandas skills

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsbombpy import sb
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## What is Logistic Regression?

Logistic Regression is a **classification algorithm** that predicts **probabilities** between 0 and 1.

### The Sigmoid Function

The key to logistic regression is the **sigmoid function**, which "squashes" any number into a probability:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Where:
- $z$ = a "score" calculated from your features
- $\sigma(z)$ = probability between 0 and 1

**Properties:**
- Very negative scores → probability close to 0
- Very positive scores → probability close to 1  
- Score of 0 → probability of 0.5

In [ ]:
# Visualize the sigmoid function
z = np.linspace(-10, 10, 100)
sigmoid = 1 / (1 + np.exp(-z))

plt.figure(figsize=(10, 6))
plt.plot(z, sigmoid, linewidth=2)
plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Threshold = 0.5')
plt.axvline(x=0, color='g', linestyle='--', alpha=0.5, label='Score = 0')
plt.xlabel('Score (z)', fontsize=12)
plt.ylabel('Probability', fontsize=12)
plt.title('The Sigmoid Function', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Building an Expected Goals (xG) Model

Let's build a model to predict the probability that a shot will result in a goal.

### Step 1: Load the Data

We'll use the 2019 UEFA Champions League Final between Liverpool and Tottenham.

In [ ]:
# Load Champions League Final 2019
matches = sb.matches(competition_id=16, season_id=4)
cl_final = matches[matches['match_id'] == 22912]

print("Match Information:")
print(f"{cl_final['home_team'].values[0]} vs {cl_final['away_team'].values[0]}")
print(f"Score: {cl_final['home_score'].values[0]}-{cl_final['away_score'].values[0]}")

In [ ]:
# Get event data and filter for shots
events = sb.events(match_id=22912)
shots = events[events['type'] == 'Shot'].copy()

print(f"Total shots: {len(shots)}")
print(f"\nShot outcomes:")
print(shots['shot_outcome_name'].value_counts())

### Step 2: Feature Engineering

We need to create features that help predict if a shot will be a goal. Two important features are:

1. **Distance to Goal** - How far from the goal was the shot taken?
2. **Angle to Goal** - What angle did the shooter have to the goal?

In StatsBomb data:
- Goal is located at coordinates (120, 40)
- Goal width is 7.32 meters

In [ ]:
# Create binary target variable
shots['goal'] = np.where(shots['shot_outcome_name'] == 'Goal', 1, 0)

# Extract x and y coordinates from location
shots['x'] = shots['location'].apply(lambda loc: loc[0])
shots['y'] = shots['location'].apply(lambda loc: loc[1])

# Calculate distance to goal (center at 120, 40)
shots['distance_to_goal'] = np.sqrt(
    (120 - shots['x'])**2 + (40 - shots['y'])**2
)

# Calculate angle to goal (using trigonometry)
goal_width = 7.32
shots['angle_to_goal'] = np.arctan(
    goal_width * (120 - shots['x']) / 
    ((120 - shots['x'])**2 + (40 - shots['y'])**2 - (goal_width/2)**2)
)
shots['angle_to_goal'] = np.degrees(shots['angle_to_goal'])

print("Engineered features:")
print(shots[['x', 'y', 'distance_to_goal', 'angle_to_goal', 'goal']].head(10))

### Step 3: Visualize Features vs Outcome

In [ ]:
# Visualize relationship between features and goals
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distance vs Goal
axes[0].scatter(shots['distance_to_goal'], shots['goal'], alpha=0.6)
axes[0].set_xlabel('Distance to Goal (meters)')
axes[0].set_ylabel('Goal (1=Yes, 0=No)')
axes[0].set_title('Distance vs Goal Outcome')

# Angle vs Goal
axes[1].scatter(shots['angle_to_goal'], shots['goal'], alpha=0.6)
axes[1].set_xlabel('Angle to Goal (degrees)')
axes[1].set_ylabel('Goal (1=Yes, 0=No)')
axes[1].set_title('Angle vs Goal Outcome')

plt.tight_layout()
plt.show()

**Observations:**
- Goals tend to come from **shorter distances**
- Goals tend to come from **wider angles** (better view of goal)

### Step 4: Prepare Data for Training

In [ ]:
# Select features and target
features = ['distance_to_goal', 'angle_to_goal']
X = shots[features]
y = shots['goal']

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"\nFeature names: {features}")

**Why split the data?**
- Train on **training set** to learn patterns
- Test on **test set** to evaluate on unseen data
- This gives an honest assessment of performance

### Step 5: Train the Logistic Regression Model

In [ ]:
# Initialize and train the model
log_reg = LogisticRegression()
log_reg.fit(X_train, y_train)

print("Model trained successfully!")
print(f"\nModel coefficients: {log_reg.coef_[0]}")
print(f"Model intercept: {log_reg.intercept_[0]}")

### Step 6: Interpret the Coefficients

The coefficients tell us how each feature affects the probability of scoring:

- **Negative coefficient for distance**: As distance increases, probability of goal **decreases** ✓
- **Positive coefficient for angle**: As angle increases, probability of goal **increases** ✓

This matches our soccer intuition!

In [ ]:
# Display coefficients in a readable format
coef_df = pd.DataFrame({
    'Feature': features,
    'Coefficient': log_reg.coef_[0],
    'Effect': ['Negative (further = less likely)', 'Positive (wider = more likely)']
})

print("\nCoefficient Interpretation:")
print(coef_df.to_string(index=False))

### Step 7: Make Predictions

In [ ]:
# Get predicted probabilities for test set
y_pred_proba = log_reg.predict_proba(X_test)[:, 1]

# Get binary predictions (using 0.5 threshold)
y_pred = log_reg.predict(X_test)

# Show some predictions
results = pd.DataFrame({
    'Distance': X_test['distance_to_goal'].values,
    'Angle': X_test['angle_to_goal'].values,
    'Actual': y_test.values,
    'Predicted Probability': y_pred_proba,
    'Predicted Class': y_pred
})

print("Sample Predictions:")
print(results.head(10).to_string(index=False))

### Step 8: Predict for New Shots

Let's predict xG for hypothetical shots:

In [ ]:
# Create hypothetical shots
new_shots = pd.DataFrame({
    'distance_to_goal': [5, 10, 20, 30],
    'angle_to_goal': [30, 25, 15, 10],
    'description': [
        'Close range, wide angle',
        'Inside box, good angle',
        'Edge of box, narrow angle',
        'Long range, very narrow angle'
    ]
})

# Predict xG
new_shots['xG'] = log_reg.predict_proba(
    new_shots[['distance_to_goal', 'angle_to_goal']]
)[:, 1]

print("\nExpected Goals for New Shots:")
print(new_shots[['description', 'distance_to_goal', 'angle_to_goal', 'xG']].to_string(index=False))

## Summary

In this notebook, we:

1. ✅ Learned how **logistic regression** uses the sigmoid function
2. ✅ Built an **Expected Goals (xG) model**
3. ✅ Engineered features: **distance** and **angle** to goal
4. ✅ Trained the model and **interpreted coefficients**
5. ✅ Made **predictions** for new shots

## Key Takeaways

- Logistic regression predicts **probabilities** (0 to 1)
- Coefficients show **feature importance** and **direction of effect**
- xG models help quantify **shot quality**
- Always **split data** into train/test sets

## Next Steps

In the next notebook, we'll learn how to **evaluate** our model's performance using metrics like accuracy, precision, recall, and AUC!

## Practice Exercises

1. **Add More Features**: Try adding `shot_technique_name` or `shot_body_part_name` as features
2. **Different Match**: Build an xG model for a different match
3. **Visualize Decision Boundary**: Plot the model's decision boundary in 2D space
4. **Feature Importance**: Which feature has a stronger effect on xG?
5. **Manual Calculation**: Calculate the xG for a shot at distance=15, angle=20 using the coefficients